# 03 Run 405841 Segments (Part 2 / RegGS)

Goal:

1. run RegGS on `405841` prepared segmented scenes;
2. select one scene manually (`A` / `B` / `C`) and run only that one;
3. keep the infer / refine / metric pipeline unchanged;
4. keep a clean notebook entry for reproducible single-scene runs.

This notebook consumes data prepared by `01c_prepare_reggs_scene_405841_3segments.ipynb`.

In [13]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [14]:
from pathlib import Path
import json
import subprocess
import shutil
import copy
import yaml
from PIL import Image

CV_ROOT = Path('~/CV_Project').expanduser()
REGGS_ROOT = CV_ROOT / 'third_party' / 'RegGS'
REGGS_PYTHON = Path('/home/bzhang512/miniconda3/envs/reggs/bin/python')
PART2_ROOT = CV_ROOT / 'part2'
CONFIG_ROOT = PART2_ROOT / 'configs'
OUTPUT_ROOT = CV_ROOT / 'output' / 'part2'
RESULTS_ROOT = CV_ROOT / 'results' / 'part2'

SCENE_GROUP_ROOT = CV_ROOT / 'dataset' / '405841' / 'part2' / 'reggs_405841_3seg_25_70'
SCENE_OPTIONS = {
    'A': 'scene_A',
    'B': 'scene_B',
    'C': 'scene_C',
}

for d in [CONFIG_ROOT, OUTPUT_ROOT, RESULTS_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print('REGGS_ROOT =', REGGS_ROOT)
print('REGGS_PYTHON =', REGGS_PYTHON)
print('SCENE_GROUP_ROOT =', SCENE_GROUP_ROOT)
print('SCENE_OPTIONS =', SCENE_OPTIONS)

REGGS_ROOT = /home/bzhang512/CV_Project/third_party/RegGS
REGGS_PYTHON = /home/bzhang512/miniconda3/envs/reggs/bin/python
SCENE_GROUP_ROOT = /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70
SCENE_OPTIONS = {'A': 'scene_A', 'B': 'scene_B', 'C': 'scene_C'}


## 1. Run configuration (manual scene selection)


In [16]:
# Manual scene switch: choose one from {'A', 'B', 'C'} and run only that scene.
SCENE_KEY = 'C'

assert SCENE_KEY in SCENE_OPTIONS, f'Invalid SCENE_KEY={SCENE_KEY}; choose one of {list(SCENE_OPTIONS.keys())}'
SCENE_NAME = SCENE_OPTIONS[SCENE_KEY]
SCENE_ROOT = SCENE_GROUP_ROOT / SCENE_NAME
SCENE_PARENT = SCENE_ROOT.parent

# Keep the same pipeline knobs; only scene address/selection is changed.
FORMAL_SAMPLE_RATE = 30
FORMAL_N_VIEWS = 12
FORMAL_NEW_SUBMAP_EVERY = 2

# Aligner stability knobs (recommended start values; comments show RegGS defaults)
ALIGNER_ITERATIONS = 300         # default: 200
ALIGNER_CAM_ROT_LR = 1.0e-4      # default: 2.0e-4
ALIGNER_CAM_TRANS_LR = 1.0e-3    # default: 2.0e-3
ALIGNER_FILTER_ALPHA = True      # default: False
ALIGNER_ALPHA_THRE = 0.95        # default: 0.98
ALIGNER_MASK_INVALID_DEPTH = True  # default: False
ALIGNER_SCALE_MIN = 0.10         # default: 0.05
ALIGNER_SCALE_MAX = 10.0         # default: 20.0
ALIGNER_DEPTH_VALID_MIN = 0.10   # default: 0.05
ALIGNER_SCALE_INIT_FALLBACK = 1.0  # default: 1.0
ALIGNER_MW2_WARMUP_ITERS = 60    # default: 20

# Human-readable profile label for path/result separation when changing aligner knobs
RUN_PROFILE_TAG = 'stable_v1'

# 405841 has no dedicated ckpt: manual override > existing candidate fallback
FORMAL_NOPO_CHECKPOINT = 'dl3dv'  # e.g. 'dl3dv' / 're10k' / 'acid' / None
NOPO_CHECKPOINT_CANDIDATES = ['dl3dv', 're10k', 'acid']
VALID_NOPO_CHECKPOINTS = {'re10k', 'dl3dv', 'acid'}
NOPO_CHECKPOINT_TO_FILE = {
    're10k': 're10k.ckpt',
    'dl3dv': 'mixRe10kDl3dv.ckpt',
    'acid': 'acid.ckpt',
}

if not NOPO_CHECKPOINT_CANDIDATES:
    raise ValueError('NOPO_CHECKPOINT_CANDIDATES cannot be empty.')

for ckpt_name in NOPO_CHECKPOINT_CANDIDATES:
    if ckpt_name not in VALID_NOPO_CHECKPOINTS:
        raise ValueError(
            f'Invalid candidate ckpt={ckpt_name}; valid options are {sorted(VALID_NOPO_CHECKPOINTS)}'
        )

if FORMAL_NOPO_CHECKPOINT is not None:
    if FORMAL_NOPO_CHECKPOINT not in VALID_NOPO_CHECKPOINTS:
        raise ValueError(
            f'Invalid FORMAL_NOPO_CHECKPOINT={FORMAL_NOPO_CHECKPOINT}; '
            f'valid options are {sorted(VALID_NOPO_CHECKPOINTS)}'
        )
    RESOLVED_NOPO_CHECKPOINT = FORMAL_NOPO_CHECKPOINT
    nopo_source = 'manual'
else:
    existing_candidates = [
        ck for ck in NOPO_CHECKPOINT_CANDIDATES
        if (REGGS_ROOT / 'pretrained_weights' / NOPO_CHECKPOINT_TO_FILE[ck]).exists()
    ]
    if existing_candidates:
        RESOLVED_NOPO_CHECKPOINT = existing_candidates[0]
        nopo_source = 'existing_candidate_fallback'
    else:
        RESOLVED_NOPO_CHECKPOINT = NOPO_CHECKPOINT_CANDIDATES[0]
        nopo_source = 'candidate_fallback'
        print(
            f'[WARN] none of NOPO_CHECKPOINT_CANDIDATES exists under pretrained_weights; '
            f'fallback to {RESOLVED_NOPO_CHECKPOINT}.'
        )

NOPO_WEIGHT_PATH = REGGS_ROOT / 'pretrained_weights' / NOPO_CHECKPOINT_TO_FILE[RESOLVED_NOPO_CHECKPOINT]
if not NOPO_WEIGHT_PATH.exists():
    print(f'[WARN] selected ckpt file not found: {NOPO_WEIGHT_PATH}')
else:
    print(f'[OK] selected ckpt file found: {NOPO_WEIGHT_PATH}')

profile_tag = RUN_PROFILE_TAG.strip() if RUN_PROFILE_TAG.strip() else 'default'
RUN_TAG = (
    f'reggs_405841_{SCENE_NAME}_{RESOLVED_NOPO_CHECKPOINT}-ckpt_'
    f'sr{FORMAL_SAMPLE_RATE}_nv{FORMAL_N_VIEWS}_sm{FORMAL_NEW_SUBMAP_EVERY}_{profile_tag}'
)
RUN_OUTPUT = OUTPUT_ROOT / '405841' / RUN_TAG
CONFIG_PATH = CONFIG_ROOT / f'{RUN_TAG}.yaml'

print('SCENE_KEY =', SCENE_KEY)
print('SCENE_NAME =', SCENE_NAME)
print('SCENE_ROOT =', SCENE_ROOT)
print('NOPO_CHECKPOINT_CANDIDATES =', NOPO_CHECKPOINT_CANDIDATES)
print('RESOLVED_NOPO_CHECKPOINT =', RESOLVED_NOPO_CHECKPOINT)
print('nopo selection source =', nopo_source)
print('RUN_PROFILE_TAG =', profile_tag)
print('RUN_TAG =', RUN_TAG)
print('RUN_OUTPUT =', RUN_OUTPUT)
print('CONFIG_PATH =', CONFIG_PATH)
print('ALIGNER knobs =', {
    'iterations': ALIGNER_ITERATIONS,
    'cam_rot_lr': ALIGNER_CAM_ROT_LR,
    'cam_trans_lr': ALIGNER_CAM_TRANS_LR,
    'filter_alpha': ALIGNER_FILTER_ALPHA,
    'alpha_thre': ALIGNER_ALPHA_THRE,
    'mask_invalid_depth': ALIGNER_MASK_INVALID_DEPTH,
    'scale_min': ALIGNER_SCALE_MIN,
    'scale_max': ALIGNER_SCALE_MAX,
    'depth_valid_min': ALIGNER_DEPTH_VALID_MIN,
    'scale_init_fallback': ALIGNER_SCALE_INIT_FALLBACK,
    'mw2_warmup_iters': ALIGNER_MW2_WARMUP_ITERS,
})

[OK] selected ckpt file found: /home/bzhang512/CV_Project/third_party/RegGS/pretrained_weights/mixRe10kDl3dv.ckpt
SCENE_KEY = C
SCENE_NAME = scene_C
SCENE_ROOT = /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70/scene_C
NOPO_CHECKPOINT_CANDIDATES = ['dl3dv', 're10k', 'acid']
RESOLVED_NOPO_CHECKPOINT = dl3dv
nopo selection source = manual
RUN_PROFILE_TAG = stable_v1
RUN_TAG = reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_stable_v1
RUN_OUTPUT = /home/bzhang512/CV_Project/output/part2/405841/reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_stable_v1
CONFIG_PATH = /home/bzhang512/CV_Project/part2/configs/reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_stable_v1.yaml
ALIGNER knobs = {'iterations': 300, 'cam_rot_lr': 0.0001, 'cam_trans_lr': 0.001, 'filter_alpha': True, 'alpha_thre': 0.95, 'mask_invalid_depth': True, 'scale_min': 0.1, 'scale_max': 10.0, 'depth_valid_min': 0.1, 'scale_init_fallback': 1.0, 'mw2_warmup_iters': 60}


## 2. Basic scene checks


In [17]:
assert SCENE_GROUP_ROOT.exists(), f'Missing segmented scene root: {SCENE_GROUP_ROOT}'
assert SCENE_ROOT.exists(), f'Missing selected scene root: {SCENE_ROOT}'
assert (SCENE_ROOT / 'images').exists(), 'Missing images dir'
assert (SCENE_ROOT / 'intrinsics.json').exists(), 'Missing intrinsics.json'
assert (SCENE_ROOT / 'cameras.json').exists(), 'Missing cameras.json'

cameras = json.loads((SCENE_ROOT / 'cameras.json').read_text(encoding='utf-8'))
intrinsics = json.loads((SCENE_ROOT / 'intrinsics.json').read_text(encoding='utf-8'))
print('selected scene =', SCENE_NAME)
print('num frames =', len(cameras))
print('intrinsics =', intrinsics)

selected scene = scene_C
num frames = 129
intrinsics = {'fx': 1.0764049814673433, 'fy': 1.6146074722010149, 'cx': 0.4950787903203501, 'cy': 0.5009273860525132}


## 3. Build formal config


In [18]:
base_cfg_path = REGGS_ROOT / 'config' / 're10k.yaml'
base_cfg = yaml.safe_load(base_cfg_path.read_text(encoding='utf-8'))

cfg = copy.deepcopy(base_cfg)
cfg['dataset_name'] = 're10k'
cfg['n_views'] = int(FORMAL_N_VIEWS)
cfg['sample_rate'] = int(FORMAL_SAMPLE_RATE)
cfg['new_submap_every'] = int(FORMAL_NEW_SUBMAP_EVERY)
cfg['nopo_checkpoint'] = RESOLVED_NOPO_CHECKPOINT
cfg['frame_limit'] = -1

cfg['data']['input_path'] = str(SCENE_PARENT)
cfg['data']['scene_name'] = SCENE_NAME
cfg['data']['output_path'] = str(RUN_OUTPUT)

image_paths = sorted((SCENE_ROOT / 'images').glob('*.png'))
assert image_paths, f'No images found in {SCENE_ROOT / "images"}'
with Image.open(image_paths[0]) as im:
    img_w, img_h = im.size

cfg['cam']['H'] = int(img_h)
cfg['cam']['W'] = int(img_w)
cfg['cam']['fx'] = float(intrinsics['fx'])
cfg['cam']['fy'] = float(intrinsics['fy'])
cfg['cam']['cx'] = float(intrinsics['cx'])
cfg['cam']['cy'] = float(intrinsics['cy'])
cfg['cam']['depth_scale'] = 5000.0

cfg.setdefault('aligner', {})
cfg['aligner']['iterations'] = int(ALIGNER_ITERATIONS)
cfg['aligner']['cam_rot_lr'] = float(ALIGNER_CAM_ROT_LR)
cfg['aligner']['cam_trans_lr'] = float(ALIGNER_CAM_TRANS_LR)
cfg['aligner']['filter_alpha'] = bool(ALIGNER_FILTER_ALPHA)
cfg['aligner']['alpha_thre'] = float(ALIGNER_ALPHA_THRE)
cfg['aligner']['mask_invalid_depth'] = bool(ALIGNER_MASK_INVALID_DEPTH)
cfg['aligner']['scale_min'] = float(ALIGNER_SCALE_MIN)
cfg['aligner']['scale_max'] = float(ALIGNER_SCALE_MAX)
cfg['aligner']['depth_valid_min'] = float(ALIGNER_DEPTH_VALID_MIN)
cfg['aligner']['scale_init_fallback'] = float(ALIGNER_SCALE_INIT_FALLBACK)
cfg['aligner']['mw2_warmup_iters'] = int(ALIGNER_MW2_WARMUP_ITERS)

CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('detected image size =', (img_w, img_h))
print('using intrinsics =', intrinsics)
print('using checkpoint =', RESOLVED_NOPO_CHECKPOINT)
print(CONFIG_PATH.read_text(encoding='utf-8'))


detected image size = (960, 640)
using intrinsics = {'fx': 1.0764049814673433, 'fy': 1.6146074722010149, 'cx': 0.4950787903203501, 'cy': 0.5009273860525132}
using checkpoint = dl3dv
dataset_name: re10k
checkpoint_path: null
frame_limit: -1
seed: 0
nopo_enable: true
n_views: 12
sample_rate: 30
nopo_checkpoint: dl3dv
new_submap_every: 2
aligner:
  iterations: 300
  cam_rot_lr: 0.0001
  cam_trans_lr: 0.001
  filter_alpha: true
  alpha_thre: 0.95
  soft_alpha: true
  mask_invalid_depth: true
  key_frame_pose: nopo
  scale_min: 0.1
  scale_max: 10.0
  depth_valid_min: 0.1
  scale_init_fallback: 1.0
  mw2_warmup_iters: 60
data:
  input_path: /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70
  output_path: /home/bzhang512/CV_Project/output/part2/405841/reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_stable_v1
  scene_name: scene_C
cam:
  H: 640
  W: 960
  fx: 1.0764049814673433
  fy: 1.6146074722010149
  cx: 0.4950787903203501
  cy: 0.5009273860525132
  crop_edge: 0
  dept

## 4. Optional cleanup


In [19]:
RESET_RUN_OUTPUT = True
if RESET_RUN_OUTPUT and RUN_OUTPUT.exists():
    shutil.rmtree(RUN_OUTPUT)
    print('removed old run output:', RUN_OUTPUT)
else:
    print('RESET_RUN_OUTPUT =', RESET_RUN_OUTPUT)


RESET_RUN_OUTPUT = True


## 5. Formal split preview


In [20]:
import numpy as np
frame_ids = np.arange(len(cameras))
test_ids = frame_ids[int(FORMAL_SAMPLE_RATE / 2)::FORMAL_SAMPLE_RATE]
remain_ids = np.array([i for i in frame_ids if i not in test_ids])
train_ids = remain_ids[np.linspace(0, remain_ids.shape[0] - 1, FORMAL_N_VIEWS).astype(int)]

print('train_ids =', train_ids.tolist())
print('test_ids =', test_ids.tolist())
print('num_train =', len(train_ids))
print('num_test =', len(test_ids))


train_ids = [0, 11, 23, 34, 47, 58, 69, 81, 93, 104, 116, 128]
test_ids = [15, 45, 75, 105]
num_train = 12
num_test = 4


## 6. Run infer


In [21]:
RUN_INFER = True
infer_cmd = [str(REGGS_PYTHON), 'run_infer.py', str(CONFIG_PATH)]
print('infer_cmd =', ' '.join(infer_cmd))

if RUN_INFER:
    proc = subprocess.run(
        infer_cmd,
        cwd=str(REGGS_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)
    print('infer returncode =', proc.returncode)
else:
    print('Set RUN_INFER=True to launch formal infer.')


infer_cmd = /home/bzhang512/miniconda3/envs/reggs/bin/python run_infer.py /home/bzhang512/CV_Project/part2/configs/reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_stable_v1.yaml
Using mixRe10kDl3dv.ckpt
/home/bzhang512/CV_Project/third_party/RegGS/src/utils/nopo_utils.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any us

## 7. Run refine


In [22]:
RUN_REFINE = True
refine_cmd = [str(REGGS_PYTHON), 'run_refine.py', '--checkpoint_path', str(RUN_OUTPUT), '--config_path', str(CONFIG_PATH)]
print('refine_cmd =', ' '.join(refine_cmd))

if RUN_REFINE:
    proc = subprocess.run(
        refine_cmd,
        cwd=str(REGGS_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)
    print('refine returncode =', proc.returncode)
else:
    print('Set RUN_REFINE=True after infer completes.')


refine_cmd = /home/bzhang512/miniconda3/envs/reggs/bin/python run_refine.py --checkpoint_path /home/bzhang512/CV_Project/output/part2/405841/reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_stable_v1 --config_path /home/bzhang512/CV_Project/part2/configs/reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_stable_v1.yaml
/home/bzhang512/CV_Project/third_party/RegGS/run_refine.py:61: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via

## 8. Run metric


In [23]:
RUN_METRIC = True
METRIC_TEST_PROTOCOL = 'sampled_test'  # options: 'sampled_test' / 'all_non_train'
METRIC_OUTPUT_TAG = ''  # optional, e.g. 'all_non_train_v1'

if METRIC_TEST_PROTOCOL not in {'sampled_test', 'all_non_train'}:
    raise ValueError(
        f"Invalid METRIC_TEST_PROTOCOL={METRIC_TEST_PROTOCOL}; "
        "valid options are {'sampled_test', 'all_non_train'}"
    )

metric_cmd = [
    str(REGGS_PYTHON),
    'run_metric.py',
    '--checkpoint_path', str(RUN_OUTPUT),
    '--config_path', str(CONFIG_PATH),
    '--test_protocol', METRIC_TEST_PROTOCOL,
]
if METRIC_OUTPUT_TAG:
    metric_cmd += ['--test_output_tag', METRIC_OUTPUT_TAG]

print('metric_cmd =', ' '.join(metric_cmd))
print('METRIC_TEST_PROTOCOL =', METRIC_TEST_PROTOCOL)
print('METRIC_OUTPUT_TAG =', METRIC_OUTPUT_TAG)

if RUN_METRIC:
    proc = subprocess.run(
        metric_cmd,
        cwd=str(REGGS_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)
    print('metric returncode =', proc.returncode)
else:
    print('Set RUN_METRIC=True after refine completes.')


metric_cmd = /home/bzhang512/miniconda3/envs/reggs/bin/python run_metric.py --checkpoint_path /home/bzhang512/CV_Project/output/part2/405841/reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_stable_v1 --config_path /home/bzhang512/CV_Project/part2/configs/reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_stable_v1.yaml --test_protocol sampled_test
METRIC_TEST_PROTOCOL = sampled_test
METRIC_OUTPUT_TAG = 
[Evaluator] Using gaussian file: /home/bzhang512/CV_Project/output/part2/405841/reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_stable_v1/gaussians/global_refined_gs.ply
Training Frames: [  0  11  23  34  47  58  69  81  93 104 116 128]
Eval Frames (sampled_test): [ 15  45  75 105]
Evaluating train render...
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/bzhang512/miniconda3/envs/reggs/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' i

## 9. Output check


In [24]:
print('run output exists =', RUN_OUTPUT.exists())
if RUN_OUTPUT.exists():
    for p in sorted(RUN_OUTPUT.iterdir())[:30]:
        print(' -', p.name)
else:
    print('No run output yet.')


run output exists = True
 - ate_aligned.json
 - config.yaml
 - estimated_c2w.ckpt
 - eval_test.json
 - eval_train.json
 - eval_traj.png
 - gaussians
 - gt
 - submaps
 - test
 - train
 - vis_align
 - vis_integrate
